# Tree Agent Test


In [1]:
# imports and config
import csv
import logging
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

cwd = Path.cwd()
root = cwd if (cwd / "src").exists() else cwd.parents[1]
if str(root) not in sys.path:
    sys.path.append(str(root))

from src.llm.llm import OpenAILLM
from src.preprocessing.chunker import ParagraphChunker
from src.preprocessing.embedder import HTTPEmbedder
from src.preprocessing.reader import Reader
from src.tree_rag.agent import TreeAgent
from src.tree_rag.index import TreeIndex
from src.tree_rag.preprocessing import TreePreprocessor

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logger = logging.getLogger("tree_agent_test")


base_url = os.getenv("BASE_OPEN_AI_URL")
api_key = os.getenv("OPEN_AI_KEY")
client = OpenAI(api_key=api_key, base_url=base_url)
reader = Reader(logger=logger)
llm = OpenAILLM(client=client, model_name="gpt-5.4-mini")
preprocessor = TreePreprocessor(llm=llm, logger=logger)
chunker = ParagraphChunker(
    logger=logger,
    max_length=490,
    overlap_sentences=1,
)
embedder = HTTPEmbedder(logger=None)

index_dir = root / "src" / "tree_rag" / "test_index"
tree_index = TreeIndex(
    index_dir=index_dir,
    reader=reader,
    preprocessor=preprocessor,
    chunker=chunker,
    embedder=embedder,
    logger=logger,
)

agent = TreeAgent(
    client=client,
    model_name="gpt-5.4",
    tree_index=tree_index,
    max_iters=30,
    top_k=5,
    logger=logger,
)


/Users/danilaandreev/Documents/HSE/degree/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# print(tree_index.llm_view())

In [ ]:
benchmark_path = root / "src" / "benchmark" / "results" / "generation_benchmark_v1_20260402_212121" / "benchmark.csv"
with benchmark_path.open(encoding="utf-8", newline="") as f:
    reader_csv = csv.DictReader(f)
    first_sample = next(reader_csv)

question = first_sample["question"]
source_path = first_sample["source_path"]
question, source_path

('Какие грунты основания, помимо слабых и просадочных, необходимо учитывать при расчете осадок из-за оттаивания?',
 '../../../data/raw_data/4293732009.pdf')

In [4]:
first_sample

{'sample_id': 'factual-001-02572f07',
 'question_type': 'factual',
 'question': 'Какие грунты основания, помимо слабых и просадочных, необходимо учитывать при расчете осадок из-за оттаивания?',
 'gold_answer': 'Вечномерзлые грунты.',
 'source_path': '../../../data/raw_data/4293732009.pdf',
 'page_number': '15',
 'source_excerpt': '',
 'judge_schema_name': 'FactualJudgeResult',
 'criteria': '{"atomic_facts": ["Вечномерзлые грунты."]}',
 'metadata': '{"generation_model_name": "manual_curation_from_список_вопросов.txt", "source_characters": 0}'}

In [15]:
# run agent
answer = agent.answer("Чем отличаются требования к выходам из встроенного теплового пункта при длине помещения 12 м и менее, если он расположен менее чем в 12 м от выхода из здания, и при такой же длине, но на расстоянии более 12 м от выхода?")
print(answer)


==================== Iter 0 ====================

Question:
Чем отличаются требования к выходам из встроенного теплового пункта при длине помещения 12 м и менее, если он расположен менее чем в 12 м от выхода из здания, и при такой же длине, но на расстоянии более 12 м от выхода?

Plan:
<empty>

Goal:
<empty>

Facts:
<empty>

Candidate Documents:
<empty>


LLM Step:
- current_plan:
[todo] Найти норму о количестве и расположении выходов из встроенного теплового пункта
[todo] Проверить условие для помещения длиной 12 м и менее при расстоянии до выхода из здания менее 12 м
[todo] Проверить условие для помещения той же длины при расстоянии до выхода из здания более 12 м
[todo] Сформулировать различие требований
- current_goal: выходы из встроенного теплового пункта длина помещения 12 м и менее расстояние до выхода из здания 12 м
Search goal: выходы из встроенного теплового пункта длина помещения 12 м и менее расстояние до выхода из здания 12 м
Attention hits: { 3:[18]; 29:[4] }

==========

При длине помещения встроенного теплового пункта 12 м и менее различие такое:

- если тепловой пункт расположен менее чем в 12 м от выхода из здания — допускается один выход наружу через коридор или лестничную клетку;
- если он расположен более чем в 12 м от выхода из здания — требуется один самостоятельный выход непосредственно наружу.

То есть во втором случае выход через коридор или лестничную клетку уже недостаточен: нужен отдельный прямой выход наружу.


In [16]:
print(answer)

При длине помещения встроенного теплового пункта 12 м и менее различие такое:

- если тепловой пункт расположен менее чем в 12 м от выхода из здания — допускается один выход наружу через коридор или лестничную клетку;
- если он расположен более чем в 12 м от выхода из здания — требуется один самостоятельный выход непосредственно наружу.

То есть во втором случае выход через коридор или лестничную клетку уже недостаточен: нужен отдельный прямой выход наружу.
